# Deteccion de cambio con ChangeFormer en Gaza

Este cuaderno ejecuta la inferencia sobre el par Sentinel-2 antes/despues y muestra el mapa de cambio. Requiere los pesos preentrenados de ChangeFormer (LEVIR-CD) en `checkpoints/`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.utils.config import load_config

config = load_config('configs/gaza.yaml')

In [ ]:
from src.data.prepare import load_sentinel_pair
from src.models.changeformer_wrapper import ChangeFormerWrapper
from src.inference.predict_tiles import predict_change_map

pre_rgb, post_rgb, profile = load_sentinel_pair(config)

model = ChangeFormerWrapper(
    checkpoint=config['model']['checkpoint'],
    embed_dim=config['model']['embed_dim'],
    n_class=config['model']['n_class'],
    device=config['inference']['device'],
)

mask, prob = predict_change_map(
    pre_rgb,
    post_rgb,
    model,
    tile_size=config['tiling']['size'],
    stride=config['tiling']['stride'],
    threshold=config['inference']['threshold'],
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

overlay = post_rgb.copy()
overlay[mask] = overlay[mask] * 0.4 + np.array([1.0, 0.0, 0.0]) * 0.6

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(pre_rgb); axes[0].set_title('Antes')
axes[1].imshow(post_rgb); axes[1].set_title('Despues')
axes[2].imshow(np.clip(overlay, 0, 1)); axes[2].set_title('Cambio (rojo)')
for axis in axes:
    axis.axis('off')
plt.show()

In [ ]:
from src.postprocess.vectorize import vectorize_mask
from src.utils.config import ensure_dir

out_raster = Path(config['_repo_root']) / 'outputs' / 'rasters' / 'cambio_gaza.tif'
out_geojson = Path(config['_repo_root']) / 'outputs' / 'vectors' / 'cambio_gaza.geojson'
ensure_dir(out_raster.parent)
ensure_dir(out_geojson.parent)

import rasterio
profile_out = dict(profile); profile_out.update(count=1, dtype='uint8', compress='lzw')
with rasterio.open(out_raster, 'w', **profile_out) as dst:
    dst.write(mask.astype('uint8'), 1)

n = vectorize_mask(out_raster, out_geojson, min_area=20.0, simplify=1.0)
print(f'{n} poligonos de cambio guardados.')